# Mastodon historique : analyse Spark et visualisations

Lancer d'abord Kafka et PostgreSQL, puis la collecte et le job Spark (voir `README.md`). Ce notebook lit PostgreSQL avec Spark, compare une agrégation avant/après repartition et cache, puis produit quatre visualisations. Démarrer Jupyter depuis la racine du dépôt et sélectionner l'environnement `.venv`.

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, explode, hour, lower, sum, to_date

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
load_dotenv(Path.cwd() / ".env")

spark = (
    SparkSession.builder
    .appName("MastodonHistoricalNotebook")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.5")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

jdbc_url = (
    f"jdbc:postgresql://{os.getenv('POSTGRES_HOST', 'localhost')}:"
    f"{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'mastodon_db')}"
)
jdbc_options = {
    "url": jdbc_url,
    "user": os.getenv("POSTGRES_USER", "mastodon"),
    "password": os.getenv("POSTGRES_PASSWORD", "change_me_local"),
    "driver": "org.postgresql.Driver",
}

df = (
    spark.read.format("jdbc")
    .options(**jdbc_options)
    .option("dbtable", "toots")
    .load()
    .cache()
)
toot_count = df.count()
print(f"Publications historiques disponibles : {toot_count}")

start = time.perf_counter()
df.groupBy("username").count().count()
baseline_seconds = time.perf_counter() - start

partitions = max(1, int(os.getenv("BATCH_PARTITIONS", "4")))
optimized_df = df.repartition(partitions).cache()
optimized_df.count()
start = time.perf_counter()
optimized_df.groupBy("username").count().count()
optimized_seconds = time.perf_counter() - start
print(f"Sans optimisation : {baseline_seconds:.3f} s")
print(f"Avec repartition/cache ({partitions} partitions) : {optimized_seconds:.3f} s")

daily_counts = (
    df.groupBy(to_date("created_at").alias("day"))
    .agg(count("*").alias("toot_count"))
    .orderBy("day")
)
top_hashtags = (
    df.select(explode("hashtags").alias("hashtag"))
    .filter(col("hashtag").isNotNull())
    .groupBy(lower("hashtag").alias("hashtag"))
    .agg(count("*").alias("toot_count"))
    .orderBy(col("toot_count").desc())
    .limit(15)
)
sentiment_counts = (
    spark.read.format("jdbc")
    .options(**jdbc_options)
    .option("dbtable", "toot_sentiments")
    .load()
    .groupBy("sentiment")
    .agg(count("*").alias("toot_count"))
)
user_hourly_stats = (
    spark.read.format("jdbc")
    .options(**jdbc_options)
    .option("dbtable", "user_hourly_stats")
    .load()
)
top_users = (
    user_hourly_stats.groupBy("username")
    .agg(sum("toot_count").alias("toot_count"))
    .orderBy(col("toot_count").desc())
    .limit(20)
    .select("username")
)
user_activity = (
    user_hourly_stats.join(top_users, "username")
    .withColumn("hour", hour("window_start"))
    .groupBy("username", "hour")
    .agg(sum("toot_count").alias("toot_count"))
)

display(df.select("created_at", "username", "language", "text", "hashtags").limit(10).toPandas())

In [ ]:
daily_pdf = daily_counts.toPandas()
hashtags_pdf = top_hashtags.toPandas()
sentiment_pdf = sentiment_counts.toPandas()
activity_pdf = user_activity.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Mastodon : activité et sentiment", fontsize=16)

if not sentiment_pdf.empty:
    sentiment_pdf = sentiment_pdf.set_index("sentiment").reindex(
        ["positive", "negative", "neutral"], fill_value=0
    ).reset_index()
    sns.barplot(data=sentiment_pdf, x="sentiment", y="toot_count", ax=axes[0, 0])
    axes[0, 0].set_title("Sentiment des publications en anglais")
else:
    axes[0, 0].set_title("Aucun résultat de sentiment : exécuter le batch")

if not daily_pdf.empty:
    sns.lineplot(data=daily_pdf, x="day", y="toot_count", marker="o", ax=axes[0, 1])
    axes[0, 1].set_title("Publications par jour")
    axes[0, 1].tick_params(axis="x", rotation=30)

if not hashtags_pdf.empty:
    sns.barplot(data=hashtags_pdf, y="hashtag", x="toot_count", ax=axes[1, 0])
    axes[1, 0].set_title("Top hashtags")

if not activity_pdf.empty:
    activity_matrix = activity_pdf.pivot_table(
        index="username", columns="hour", values="toot_count", fill_value=0
    )
    sns.heatmap(activity_matrix, cmap="Blues", ax=axes[1, 1])
    axes[1, 1].set_title("Activité des 20 utilisateurs les plus actifs par heure")
    axes[1, 1].set_xlabel("Heure UTC")

plt.tight_layout()
plt.show()

optimized_df.unpersist()
df.unpersist()